# $TP2$ $—$ $Préparation$ $et$ $qualité$ $des$ $sources$
**M115 · Analyse de Données Multidimensionnelles et BI — ENSAB**  
Dataset : UCI Online Retail · Outil : Python + Pandas

# 

## **Comment aller-vous résonner ?**

### 1 : Quelles anomalies devez-vous mesurer avant de modifier les données ?

Avant de nettoyer les données, il faut examiner les valeurs manquantes, les doublons exacts, les quantités négatives ou nulles, les prix négatifs ou nuls et les factures commençant par C. Il faut également vérifier les codes produits et les descriptions pour identifier les lignes potentiellement non commerciales. Ces contrôles permettent de comprendre les problèmes présents dans la source avant toute transformation.

### 2 : Pourquoi InvoiceNo + StockCode ne constitue-t-il pas une clé de ligne ?

Une même facture peut contenir plusieurs lignes concernant le même produit. Les colonnes InvoiceNo et StockCode ne permettent donc pas d'identifier chaque ligne de manière unique. Pour supprimer les doublons, il faut comparer les huit colonnes métier et supprimer uniquement les copies strictement identiques.

### 3 : Comment distinguez-vous un code non commercial d’un code commercial alphanumérique ?

La présence d'une lettre dans StockCode ne signifie pas automatiquement que le code est non commercial. Il faut examiner le code, sa description et son comportement en utilisant la référence de classification validée. Les lignes dont la classification reste incertaine doivent être marquées A_VERIFIER jusqu'à leur vérification.

### 4 : Pourquoi InvoiceNo commençant par C ne doit-il pas être supprimé sans trace ?

Les factures commençant par C correspondent généralement à des annulations dans ce dataset. Ces lignes sont exclues du périmètre FactVente du MVP, mais elles doivent être conservées dans la couche de staging ou d'audit afin de garder une trace des opérations métier.

### 5 : Que faites-vous d’un CustomerID manquant ?

Nous conserve la ligne, car l'absence d'identifiant client ne signifie pas que la vente est invalide. Le client pourra être rattaché ultérieurement à la catégorie « Client inconnu ». Une ligne ne doit pas être supprimée uniquement parce que CustomerID est manquant.

### 6 : Quels contrôles permettront de prouver l’effet de vos transformations ?

Nous comparons les résultats avant et après la préparation en examinant :
- Le nombre total de lignes.
- Le nombre de doublons exacts.
- Le nombre de valeurs manquantes.
- Le nombre de factures commençant par C.
- Le nombre de lignes avec Quantity ≤ 0.
- Le nombre de lignes avec UnitPrice ≤ 0.
- Le nombre de lignes exclues du périmètre FactVente.
- La somme de Quantity × UnitPrice, lorsque ce montant est calculé.
Ces contrôles permettent de vérifier que les transformations respectent les règles définies et que les exclusions sont justifiées et traçables.

## **Mission Technique**

## 1. Chargement des données

In [3]:
import pandas as pd
import numpy as np
from IPython.display import display

df_brut = pd.read_csv("data.csv", encoding="cp1252")

COLONNES = ["InvoiceNo", "StockCode", "Description", "Quantity",
            "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
manquantes = [c for c in COLONNES if c not in df_brut.columns]
if manquantes:
    raise ValueError(f"Colonnes manquantes : {manquantes}")

df_brut = df_brut[COLONNES].copy()
print("Dimensions initiales :", df_brut.shape)
display(df_brut.head())

Dimensions initiales : (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## 2. Profilage initial
Ces résultats constituent la référence avant toute transformation.

In [4]:
print("Types :")
display(df_brut.dtypes.rename("Type").to_frame())

print("\nValeurs manquantes :")
display(df_brut.isna().sum().rename("Nombre manquant").to_frame())

print("\nDoublons exacts sur les huit colonnes :", df_brut.duplicated(subset=COLONNES).sum())
print("\nQuantity :")
display(df_brut["Quantity"].describe())
print("\nUnitPrice :")
display(df_brut["UnitPrice"].describe())

print("\nInvoiceNo commençant par C :",
      df_brut["InvoiceNo"].astype("string").str.startswith("C", na=False).sum())
print("Quantity <= 0 :", (df_brut["Quantity"] <= 0).sum())
print("UnitPrice <= 0 :", (df_brut["UnitPrice"] <= 0).sum())
print("CustomerID manquant :", df_brut["CustomerID"].isna().sum())

Types :


,Type
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,object
UnitPrice,float64
CustomerID,float64
Country,object



Valeurs manquantes :


,Nombre manquant
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0



Doublons exacts sur les huit colonnes : 5268

Quantity :


count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64


UnitPrice :


count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64


InvoiceNo commençant par C : 9288
Quantity <= 0 : 10624
UnitPrice <= 0 : 2517
CustomerID manquant : 135080


###  Pourquoi contrôler les données avant transformation ?
le contrôle initial fournit une référence mesurable. Il permet de comparer les résultats après préparation, de justifier les changements et de vérifier qu'aucune information n'a été supprimée sans raison.

## 3. Tableau de contrôle initial

In [5]:
def indicateurs(df):
    montant = pd.to_numeric(df["Quantity"], errors="coerce") * pd.to_numeric(df["UnitPrice"], errors="coerce")
    return {
        "Nombre de lignes": len(df),
        "Doublons exacts (8 colonnes)": int(df.duplicated(subset=COLONNES).sum()),
        "CustomerID manquant": int(df["CustomerID"].isna().sum()),
        "InvoiceNo commençant par C": int(df["InvoiceNo"].astype("string").str.startswith("C", na=False).sum()),
        "Quantity <= 0": int((pd.to_numeric(df["Quantity"], errors="coerce") <= 0).sum()),
        "UnitPrice <= 0": int((pd.to_numeric(df["UnitPrice"], errors="coerce") <= 0).sum()),
        "Quantity min": pd.to_numeric(df["Quantity"], errors="coerce").min(),
        "Quantity max": pd.to_numeric(df["Quantity"], errors="coerce").max(),
        "UnitPrice min": pd.to_numeric(df["UnitPrice"], errors="coerce").min(),
        "UnitPrice max": pd.to_numeric(df["UnitPrice"], errors="coerce").max(),
        "Somme Quantity × UnitPrice": float(montant.sum())
    }

initial = indicateurs(df_brut)
display(pd.DataFrame({"Indicateur": initial.keys(), "Valeur initiale": initial.values()}))

,Indicateur,Valeur initiale
0,Nombre de lignes,541909.000
1,Doublons exacts (8 colonnes),5268.000
2,CustomerID manquant,135080.000
3,InvoiceNo commençant par C,9288.000
4,Quantity <= 0,10624.000
5,UnitPrice <= 0,2517.000
6,Quantity min,-80995.000
7,Quantity max,80995.000
8,UnitPrice min,-11062.060
9,UnitPrice max,38970.000


## 4. Mesurer les anomalies
Aucune ligne n'est supprimée à cette étape. Les indicateurs servent à documenter les cas métier.

In [6]:
df_audit = df_brut.copy()
df_audit["Audit_Annulation"] = df_audit["InvoiceNo"].astype("string").str.startswith("C", na=False)
df_audit["Audit_Quantity_NonPositive"] = pd.to_numeric(df_audit["Quantity"], errors="coerce") <= 0
df_audit["Audit_UnitPrice_NonPositive"] = pd.to_numeric(df_audit["UnitPrice"], errors="coerce") <= 0
df_audit["Audit_CustomerID_Manquant"] = df_audit["CustomerID"].isna()
df_audit["Audit_StockCode_Alphanumerique"] = df_audit["StockCode"].astype("string").str.contains(r"[A-Za-z]", na=False)

resume_anomalies = pd.DataFrame({
    "Phénomène": ["InvoiceNo commence par C", "Quantity <= 0", "UnitPrice <= 0",
                  "CustomerID manquant", "StockCode contient une lettre"],
    "Nombre": [int(df_audit[c].sum()) for c in [
        "Audit_Annulation", "Audit_Quantity_NonPositive", "Audit_UnitPrice_NonPositive",
        "Audit_CustomerID_Manquant", "Audit_StockCode_Alphanumerique"]],
    "Décision": [
        "Exclure de FactVente MVP, conserver en audit",
        "Exclure de FactVente MVP, conserver en audit",
        "Exclure de FactVente MVP, conserver en audit",
        "Conserver ; rattachement ultérieur à Client inconnu",
        "Vérifier la référence ; ne pas exclure automatiquement"
    ]
})
display(resume_anomalies)

,Phénomène,Nombre,Décision
0,InvoiceNo commence par C,9288,"Exclure de FactVente MVP, conserver en audit"
1,Quantity <= 0,10624,"Exclure de FactVente MVP, conserver en audit"
2,UnitPrice <= 0,2517,"Exclure de FactVente MVP, conserver en audit"
3,CustomerID manquant,135080,Conserver ; rattachement ultérieur à Client in...
4,StockCode contient une lettre,54873,Vérifier la référence ; ne pas exclure automat...


### Réponses aux questions
- **Une quantité négative est-elle nécessairement une erreur ?** Non. Elle peut représenter un retour ou une annulation. Elle est exclue du périmètre FactVente MVP mais conservée en staging/audit.
- **Pourquoi tracer les InvoiceNo commençant par C ?** Ils signalent des annulations selon les règles du TP. Leur conservation en audit permet de justifier leur exclusion.
- **Que faire d'un CustomerID manquant ?** Conserver la ligne. L'absence d'identifiant client ne suffit pas à supprimer une vente.
- **Une lettre dans StockCode suffit-elle à exclure une ligne ?** Non. Un code alphanumérique peut être commercial ; il faut utiliser la référence validée, la description et le comportement de la référence.

## 5. Normalisation des textes
Les valeurs originales sont conservées dans des colonnes `*_Original`. La normalisation retire seulement les espaces au début et à la fin.

In [7]:
df_norm = df_audit.copy()
for col in ["Description", "Country", "StockCode", "InvoiceNo"]:
    df_norm[f"{col}_Original"] = df_norm[col]
    df_norm[col] = df_norm[col].astype("string").str.strip()

display(df_norm[["Description_Original", "Description", "Country_Original", "Country",
                 "StockCode_Original", "StockCode"]].head(10))

,Description_Original,Description,Country_Original,Country,StockCode_Original,StockCode
0,WHITE HANGING HEART T-LIGHT HOLDER,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom,United Kingdom,85123A,85123A
1,WHITE METAL LANTERN,WHITE METAL LANTERN,United Kingdom,United Kingdom,71053,71053
2,CREAM CUPID HEARTS COAT HANGER,CREAM CUPID HEARTS COAT HANGER,United Kingdom,United Kingdom,84406B,84406B
3,KNITTED UNION FLAG HOT WATER BOTTLE,KNITTED UNION FLAG HOT WATER BOTTLE,United Kingdom,United Kingdom,84029G,84029G
4,RED WOOLLY HOTTIE WHITE HEART.,RED WOOLLY HOTTIE WHITE HEART.,United Kingdom,United Kingdom,84029E,84029E
5,SET 7 BABUSHKA NESTING BOXES,SET 7 BABUSHKA NESTING BOXES,United Kingdom,United Kingdom,22752,22752
6,GLASS STAR FROSTED T-LIGHT HOLDER,GLASS STAR FROSTED T-LIGHT HOLDER,United Kingdom,United Kingdom,21730,21730
7,HAND WARMER UNION JACK,HAND WARMER UNION JACK,United Kingdom,United Kingdom,22633,22633
8,HAND WARMER RED POLKA DOT,HAND WARMER RED POLKA DOT,United Kingdom,United Kingdom,22632,22632
9,ASSORTED COLOUR BIRD ORNAMENT,ASSORTED COLOUR BIRD ORNAMENT,United Kingdom,United Kingdom,84879,84879


**Pourquoi conserver les valeurs originales ?** Elles permettent de comparer avant/après, de retracer les transformations et de revenir à la valeur source si une erreur est détectée.

## 6. Suppression des doublons exacts
La règle porte sur les huit colonnes métier, jamais sur `InvoiceNo + StockCode` uniquement.

In [8]:
n_avant = len(df_norm)
masque_doublons = df_norm.duplicated(subset=COLONNES, keep="first")
df_doublons_retires = df_norm.loc[masque_doublons].copy()
df_sans_doublons = df_norm.loc[~masque_doublons].copy()

print("Lignes avant :", n_avant)
print("Copies exactes retirées :", len(df_doublons_retires))
print("Lignes après :", len(df_sans_doublons))
print("Doublons exacts restants :", df_sans_doublons.duplicated(subset=COLONNES).sum())

Lignes avant : 541909
Copies exactes retirées : 5268
Lignes après : 536641
Doublons exacts restants : 0


**Pourquoi deux lignes avec le même InvoiceNo et StockCode peuvent-elles être légitimes ?** Les autres colonnes peuvent différer, par exemple la quantité, le prix ou la description. La suppression n'est autorisée que si les huit colonnes sont identiques.

## 7. Classification des lignes
Complétez `reference_classification` avec la référence officielle du cours. Sans cette référence, les lignes restent `A_VERIFIER` : aucune classification commerciale n'est inventée.

In [9]:
# Ajouter ici les règles officielles : StockCode, Description, TypeLigne.
# TypeLigne : PRODUIT_COMMERCIAL, NON_COMMERCIAL ou A_VERIFIER.
reference_classification = pd.DataFrame(columns=["StockCode", "Description", "TypeLigne"])

df_classe = df_sans_doublons.copy()
df_classe["TypeLigne"] = "A_VERIFIER"

if not reference_classification.empty:
    ref = reference_classification.copy()
    ref["StockCode"] = ref["StockCode"].astype("string").str.strip()
    ref["Description"] = ref["Description"].astype("string").str.strip()

    ref_desc = ref[ref["Description"].notna() & (ref["Description"] != "")]
    if not ref_desc.empty:
        df_classe = df_classe.merge(
            ref_desc[["StockCode", "Description", "TypeLigne"]].drop_duplicates(),
            on=["StockCode", "Description"], how="left", suffixes=("", "_ref")
        )
        df_classe["TypeLigne"] = df_classe["TypeLigne_ref"].fillna(df_classe["TypeLigne"])
        df_classe.drop(columns=["TypeLigne_ref"], inplace=True)

    ref_code = ref[ref["Description"].isna() | (ref["Description"] == "")]
    if not ref_code.empty:
        mapping = ref_code.drop_duplicates("StockCode").set_index("StockCode")["TypeLigne"]
        m = df_classe["StockCode"].isin(mapping.index)
        df_classe.loc[m, "TypeLigne"] = df_classe.loc[m, "StockCode"].map(mapping)

df_classe["TypeLigne"] = df_classe["TypeLigne"].fillna("A_VERIFIER")
df_classe["InclureFactVente"] = df_classe["TypeLigne"].eq("PRODUIT_COMMERCIAL")
display(df_classe["TypeLigne"].value_counts(dropna=False).to_frame("Nombre de lignes"))

,Nombre de lignes
TypeLigne,
A_VERIFIER,536641


## 8. Construire la source préparée
Périmètre MVP : facture non annulée, `Quantity > 0`, `UnitPrice > 0` et `TypeLigne = PRODUIT_COMMERCIAL`. Les lignes exclues restent dans la couche d'audit.

In [12]:
masque_fact = (
    ~df_classe["InvoiceNo"].astype("string").str.startswith("C", na=False)
    & (pd.to_numeric(df_classe["Quantity"], errors="coerce") > 0)
    & (pd.to_numeric(df_classe["UnitPrice"], errors="coerce") > 0)
    & (df_classe["TypeLigne"] == "PRODUIT_COMMERCIAL")
)
df_l2_factvente = df_classe.loc[masque_fact].copy()
df_l2_audit_exclus = df_classe.loc[~masque_fact].copy()

df_l2_factvente["MontantLigne"] = (
    pd.to_numeric(df_l2_factvente["Quantity"], errors="coerce")
    * pd.to_numeric(df_l2_factvente["UnitPrice"], errors="coerce")
)
print("Lignes dédoublonnées :", len(df_classe))
print("Lignes retenues FactVente MVP :", len(df_l2_factvente))
print("Lignes exclues mais conservées en audit :", len(df_l2_audit_exclus))
display(df_l2_factvente.head())

Lignes dédoublonnées : 536641
Lignes retenues FactVente MVP : 0
Lignes exclues mais conservées en audit : 536641


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Audit_Annulation,Audit_Quantity_NonPositive,Audit_UnitPrice_NonPositive,Audit_CustomerID_Manquant,Audit_StockCode_Alphanumerique,Description_Original,Country_Original,StockCode_Original,InvoiceNo_Original,TypeLigne,InclureFactVente,MontantLigne


## 9. Créer le montant dérivé MontantLigne

In [13]:
# Le montant est calculé sur la couche préparée
df_l2_factvente["MontantLigne"] = (
    df_l2_factvente["Quantity"] * df_l2_factvente["UnitPrice"]
)

print("Exemples de lignes préparées :")
display(df_l2_factvente[
    ["InvoiceNo", "StockCode", "Quantity",
     "UnitPrice", "MontantLigne"]
].head(10))

print("Montant total des lignes admissibles :",
      df_l2_factvente["MontantLigne"].sum())

Exemples de lignes préparées :


,InvoiceNo,StockCode,Quantity,UnitPrice,MontantLigne


Montant total des lignes admissibles : 0.0


## 10. Comparaison des contrôles avant/après

In [14]:
ind_init = indicateurs(df_brut)
ind_dedup = indicateurs(df_sans_doublons)
ind_fact = indicateurs(df_l2_factvente)

cles = ["Nombre de lignes", "Doublons exacts (8 colonnes)", "CustomerID manquant",
        "InvoiceNo commençant par C", "Quantity <= 0", "UnitPrice <= 0",
        "Somme Quantity × UnitPrice"]
comparaison = pd.DataFrame({
    "Contrôle": cles,
    "Avant": [ind_init[k] for k in cles],
    "Après dédoublonnage": [ind_dedup[k] for k in cles],
    "Après filtre MVP": [ind_fact[k] for k in cles]
})
comparaison["Écart dédoublonnage"] = comparaison["Après dédoublonnage"] - comparaison["Avant"]
comparaison["Écart filtre MVP"] = comparaison["Après filtre MVP"] - comparaison["Après dédoublonnage"]
display(comparaison)

,Contrôle,Avant,Après dédoublonnage,Après filtre MVP,Écart dédoublonnage,Écart filtre MVP
0,Nombre de lignes,541909.000,536641.000,0.0,-5268.00,-536641.000
1,Doublons exacts (8 colonnes),5268.000,0.000,0.0,-5268.00,0.000
2,CustomerID manquant,135080.000,135037.000,0.0,-43.00,-135037.000
3,InvoiceNo commençant par C,9288.000,9251.000,0.0,-37.00,-9251.000
4,Quantity <= 0,10624.000,10587.000,0.0,-37.00,-10587.000
5,UnitPrice <= 0,2517.000,2512.000,0.0,-5.00,-2512.000
6,Somme Quantity × UnitPrice,9747747.934,9726006.954,0.0,-21740.98,-9726006.954


## Analyse des résultats

Le tableau de contrôle met en évidence l'effet des différentes étapes de préparation appliquées à la source UCI Online Retail.
### Nombre de lignes
Le dataset contient 541 909 lignes avant toute transformation. Après la suppression des doublons exacts, il reste 536 641 lignes, soit 5 268 lignes supprimées. Cette diminution correspond uniquement aux copies strictement identiques sur les huit colonnes métier.
### Doublons exacts
L'audit initial détecte 5 268 doublons exacts. Après le dédoublonnage, ce nombre devient 0, ce qui confirme que toutes les copies techniques ont été supprimées sans utiliser InvoiceNo + StockCode comme critère unique.
### CustomerID manquants
Avant la préparation, 135 080 lignes ne possèdent pas d'identifiant client. Après le dédoublonnage, ce nombre passe à 135 037, soit une diminution de 43 lignes, ce qui montre que certaines valeurs manquantes appartenaient à des doublons supprimés. Les lignes avec un CustomerID manquant sont néanmoins conservées conformément aux règles du TP.
### Factures commençant par C
Le dataset contient 9 288 factures dont InvoiceNo commence par C. Après le dédoublonnage, il en reste 9 251, soit 37 lignes supprimées parce qu'elles étaient des doublons exacts. Ces factures représentent des annulations ou des retours et doivent être conservées dans la couche d'audit.
### Quantités inférieures ou égales à zéro
On observe 10 624 lignes avec une quantité inférieure ou égale à zéro avant la préparation. Après le dédoublonnage, ce nombre devient 10 587, soit une réduction de 37 lignes liée à la suppression des doublons.
### Prix inférieurs ou égaux à zéro
Le contrôle révèle 2 517 lignes avec un prix inférieur ou égal à zéro. Après le dédoublonnage, il reste 2 512 lignes, ce qui correspond à 5 doublons supprimés.
### Somme du montant estimé
La somme de Quantity × UnitPrice passe de 9 747 747,934 à 9 726 006,954 après le dédoublonnage, soit une diminution de 21 740,98. Cette différence provient des lignes dupliquées qui ont été supprimées.

## Interprétation générale
Les résultats montrent que la préparation des données améliore la qualité de la source sans supprimer abusivement les informations métier. Les doublons exacts sont entièrement éliminés, tandis que les annulations, les retours et les lignes dont CustomerID est absent restent documentés selon les règles du projet. La baisse du nombre de lignes et du montant total s'explique uniquement par la suppression des copies exactes.